# Prompt Lab: LilyPond Generation

## Overview

This notebook explores the prompt engineering and generation capabilities for LilyPond musical notation. It provides a framework for testing different prompt configurations and AI models to generate valid LilyPond code from natural language descriptions.

## Objectives

- **Generate LilyPond Code**: Use language models (via HuggingFace) to generate LilyPond notation based on text prompts
- **Test Different Prompts**: Experiment with various prompting strategies (zero-shot, few-shot, etc.)
- **Evaluate Quality**: Assess the quality and validity of generated LilyPond code
- **Document Results**: Save and analyze outputs for comparison

## Notebook Structure

This notebook is organized into sections that:
1. Load and configure prompts
2. Initialize the OpenAI client for HuggingFace models
3. Generate LilyPond code using specified parameters
4. Process and save results
5. Document generation metadata

## Requirements

- **Python 3.8+**
- **HF_TOKEN**: HuggingFace API token (set as environment variable)
- **Model Access**: Access to the selected model via HuggingFace router

## Key Parameters

- **Model**: Specify which model to use (default: OpenAI/GPT models via HuggingFace)
- **Temperature**: Controls randomness in generation (0.7 = balanced)
- **Max Tokens**: Maximum length of generated responses
- **Number of Runs**: How many samples to generate per prompt

## Output

Generated LilyPond code is saved to `prompt_lab_outputs/raw` for evaluation and analysis in subsequent notebooks.

In [2]:
from openai import OpenAI
from IPython.display import display, Markdown
from pathlib import Path
from datetime import datetime
import os, re, json


# Helper: extract LilyPond code only
def extract_lilypond_code(text: str) -> str:
    """Extract LilyPond code from a model reply, removing markdown fences/preambles."""
    text = re.sub(r'```(?:lilypond)?\s*\n?(.*?)```', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'^(?:Here|Here\'s|Here is|Output|LilyPond code:|The following).*?:\s*',
                  '', text, flags=re.IGNORECASE)
    return text.strip()


# Generator (saves in prompt_lab_outputs/raw/<subfolder>)
def run_generate(
    prompt_file: str | Path = "../configs/prompts/zero_shot.txt",
    *,
    model: str = "openai/gpt-oss-20b:fireworks-ai",
    num_runs: int = 2,
    max_tokens: int = 4096,
    temperature: float = 0.7,
    output_folder_name: str = "session_01"
):

    prompt_path = Path(prompt_file)
    if not prompt_path.exists():
        display(Markdown(f"**❌ Error:** Prompt file not found: `{prompt_path}`"))
        return []

    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        display(Markdown("**❌ Error:** `HF_TOKEN` environment variable not set!"))
        return []

    client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)

    # Detect base directory safely (handles notebooks too)
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path.cwd()  # fallback for Jupyter / REPL

    save_dir = base_dir / "prompt_lab_outputs" / "raw" / output_folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    prompt_text = prompt_path.read_text(encoding="utf-8").strip()
    display(Markdown(f"### 📝 Input Prompt\n```\n{prompt_text}\n```\n---"))

    all_results = []
    for i in range(1, num_runs + 1):
        display(Markdown(f"## 🔄 Run {i}/{num_runs}"))
        t0 = datetime.now()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_text}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        dt = (datetime.now() - t0).total_seconds()

        choice = resp.choices[0]
        raw_text = (choice.message.content or "") if choice.message else ""
        lp = extract_lilypond_code(raw_text)

        # Display output (no checks)
        md = f"### 🎵 Model Output\n```lilypond\n{lp}\n```\n"
        md += f"⏱️ Time: {dt:.2f}s | Tokens: {resp.usage.total_tokens if resp.usage else '—'}"
        display(Markdown(md))
        display(Markdown("---"))

        # Save both raw and extracted
        (save_dir / f"raw_{i:02d}.txt").write_text(raw_text, encoding="utf-8")
        (save_dir / f"out_{i:02d}.ly").write_text(lp, encoding="utf-8")

        # Append metadata for summary.json
        all_results.append({
            "run": i,
            "elapsed_seconds": round(dt, 3),
            "tokens_used": resp.usage.total_tokens if resp.usage else None,
            "model": model,
            "temperature": temperature,
            "prompt_file": str(prompt_path)
        })

    # Write summary.json
    summary_path = save_dir / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    display(Markdown(f"✅ Saved all outputs and summary in `{save_dir}`"))
    return all_results

# Example:
# results = run_generate(
#     "../configs/prompts/zero_shot.txt",
#     num_runs=3,
#     output_folder_name="test_batch_A"
# )

--------------------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Relative vs Absolute</h3>
</div>

## 🧪 Relative

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.

In [4]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="relative"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}
```
⏱️ Time: 7.75s | Tokens: 1231

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}
```
⏱️ Time: 7.50s | Tokens: 1210

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\relative`

--------------

## 🧪 Absolute

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.
Use only lowercase notes a–g with explicit durations (4). Every note must include an octave mark relative to middle C (e.g., c' d' e' f').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.
No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no \transpose, no variables/macros, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [5]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="absolute"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.
Use only lowercase notes a–g with explicit durations (4). Every note must include an octave mark relative to middle C (e.g., c' d' e' f').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.
No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no \transpose, no variables/macros, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
{
\clef treble
\key c \major
\time 4/4
\tempo 4 = 100
c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4 | c''4 b'4 a'4 g'4 | f'4 e'4 d'4 c'4 | c'4 e'4 g'4 b'4 | c''4 d'4 f'4 a'4 | b'4 c''4 d'4 e'4 | f'4 g'4 a'4 b'4 |
}
```
⏱️ Time: 4.48s | Tokens: 1511

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
{ \clef treble \key c \major \time 4/4 \tempo 4 = 100 c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4 | c''4 b'4 a'4 g'4 | f'4 e'4 d'4 c'4 | c'4 e'4 g'4 c''4 | c''4 g'4 e'4 c'4 | d'4 f'4 a'4 c''4 | c''4 a'4 f'4 d'4 | }
```
⏱️ Time: 8.45s | Tokens: 1857

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\absolute`

----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Single-container vs Multi-container</h3>
</div>

## 🧪 Multi-container

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly two separate \relative c' { ... } blocks, one after the other (no other top-level music).
Each \relative block must start with exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
Use only lowercase notes a–g with explicit durations (4).
Each block must contain exactly 8 bars, each bar containing 4 quarter-notes, separated by |, and ending the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before, between, or after the two \relative blocks.

In [4]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="multi_container"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly two separate \relative c' { ... } blocks, one after the other (no other top-level music).
Each \relative block must start with exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
Use only lowercase notes a–g with explicit durations (4).
Each block must contain exactly 8 bars, each bar containing 4 quarter-notes, separated by |, and ending the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before, between, or after the two \relative blocks.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
g4 f4 e4 d4 | c4 b4 a4 g4 | f4 e4 d4 c4 | b4 a4 g4 f4 | e4 d4 c4 b4 | a4 g4 f4 e4 | d4 c4 b4 a4 | g4 f4 e4 d4 |
}
```
⏱️ Time: 10.67s | Tokens: 1632

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
g4 f4 e4 d4 | c4 b4 a4 g4 | f4 e4 d4 c4 | b4 a4 g4 f4 | e4 d4 c4 b4 | a4 g4 f4 e4 | d4 c4 b4 a4 | g4 f4 e4 d4 |
}
```
⏱️ Time: 14.71s | Tokens: 2452

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\multi_container`

----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Inline variables/macros ON vs OFF</h3>
</div>

## 🧪 Inline variables/macros OFF

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must define and use simple variables or macros inside this file to organize the notes (for example, assigning note sequences to a variable and reusing them).
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.


In [11]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="Inline_variables_off"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must define and use simple variables or macros inside this file to organize the notes (for example, assigning note sequences to a variable and reusing them).
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
mySeq = { c4 d e f }
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
\mySeq | \mySeq | \mySeq | \mySeq | \mySeq | \mySeq | \mySeq | \mySeq |
}
```
⏱️ Time: 23.66s | Tokens: 3658

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
\foo = { c4 d4 e4 f4 }
\foo | \foo | \foo | \foo | \foo | \foo | \foo | \foo |
}
```
⏱️ Time: 29.97s | Tokens: 4166

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\Inline_variables_off`

---------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Expanded repeats ON vs OFF</h3>
</div>

## 🧪 Expanded repeats OFF

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use \repeat commands or similar shorthand for repeated material instead of expanding all notes manually.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars in total (after repetition expands to that length), each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.

In [4]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="expanded_repeats_off"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use \repeat commands or similar shorthand for repeated material instead of expanding all notes manually.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars in total (after repetition expands to that length), each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
\repeat 2 { c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | }
}
```
⏱️ Time: 7.78s | Tokens: 1280

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' { \key c \major \time 4/4 \tempo 4 = 100
\repeat unfold 4 {
c4 d4 e4 f4 | g4 a4 b4 c4 |
}
}
```
⏱️ Time: 21.40s | Tokens: 3298

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\expanded_repeats_off`

----------------


<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Resolve or keep \transpose</h3>
</div>

## 🧪 Keep \transpose

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use the \transpose command to shift musical material instead of writing all transposed notes explicitly.
Use only lowercase notes a–g with explicit durations (4) inside the transposed or base material.
Ensure the total written output (after transposition) equals 8 bars, each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
Keep all final resulting pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.



In [4]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="resolve_transpose_off"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use the \transpose command to shift musical material instead of writing all transposed notes explicitly.
Use only lowercase notes a–g with explicit durations (4) inside the transposed or base material.
Ensure the total written output (after transposition) equals 8 bars, each bar containing 4 quarter-notes, separated by |, and end the last bar with |.
Keep all final resulting pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
\transpose c c {
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}
}
```
⏱️ Time: 23.83s | Tokens: 3531

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond

```
⏱️ Time: 28.45s | Tokens: 4394

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\resolve_transpose_off`

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 \tuplet vs \times</h3>
</div>

## 🧪 Normalize to \tuplet a/b { … }

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use \tuplet a/b { ... }. Do not use \times.
Use only lowercase notes a–g with explicit durations (4 or 8).
Write exactly 8 bars; each bar must sum to four quarter-note beats after applying any tuplet ratios. Separate bars with | and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.

In [3]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="tuplet"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use \tuplet a/b { ... }. Do not use \times.
Use only lowercase notes a–g with explicit durations (4 or 8).
Write exactly 8 bars; each bar must sum to four quarter-note beats after applying any tuplet ratios. Separate bars with | and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond

```
⏱️ Time: 29.72s | Tokens: 4382

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond

```
⏱️ Time: 27.46s | Tokens: 4382

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\tuplet`

------------------------

## 🧪 Keep \times a/b { … }

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use \times a/b { ... }. Do not use \tuplet.
Use only lowercase notes a–g with explicit durations (4 or 8).
Write exactly 8 bars; each bar must sum to four quarter-note beats after applying any tuplet ratios. Separate bars with | and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.



In [4]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="times"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use \times a/b { ... }. Do not use \tuplet.
Use only lowercase notes a–g with explicit durations (4 or 8).
Write exactly 8 bars; each bar must sum to four quarter-note beats after applying any tuplet ratios. Separate bars with | and end the last bar with |.
Keep all pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no repeats, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond

```
⏱️ Time: 23.82s | Tokens: 4382

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
c8 d8 e8 f8 g8 a8 b8 c8 |
d8 e8 f8 g8 a8 b8 c'8 d'8 |
e8 f8 g8 a8 b8 c'8 d'8 e'8 |
\times 3/2 { c8 d8 e8 f8 } g4 |
f8 g8 a8 b8 c'8 d'8 e'8 f'8 |
g8 a8 b8 c'8 d'8 e'8 f'8 g'8 |
a8 b8 c'8 d'8 e'8 f'8 g'8 a'8 |
b8 c'8 d'8 e'8 f'8 g'8 a'8 b'8 |
}
```
⏱️ Time: 19.96s | Tokens: 3647

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\times`

-----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Expand music functions ON vs OFF</h3>
</div>

## 🧪 Expand music functions OFF

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use music functions (e.g., \tuplet, \times, \grace, \acciaccatura, \appoggiatura, \ottava, \transpose). Do not use \repeat.
Use only lowercase notes a–g with explicit durations (4). If a function requires shorter values (e.g., tuplets or grace), you may use 8 inside that function.
Write exactly 8 bars; each bar must sum to four quarter-note beats after applying any functions. Separate bars with | and end the last bar with |.
Keep all final resulting pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.


In [5]:
results = run_generate(
    "../../configs/prompts/zero_shot.txt",
    num_runs=20,
    output_folder_name="expand_music_functions_off"
)

### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\key c \major
\time 4/4
\tempo 4 = 100
You must use music functions (e.g., \tuplet, \times, \grace, \acciaccatura, \appoggiatura, \ottava, \transpose). Do not use \repeat.
Use only lowercase notes a–g with explicit durations (4). If a function requires shorter values (e.g., tuplets or grace), you may use 8 inside that function.
Write exactly 8 bars; each bar must sum to four quarter-note beats after applying any functions. Separate bars with | and end the last bar with |.
Keep all final resulting pitches within the diatonic notes of C major (no accidentals).
No rests, no chords, no text, no \score or \layout, no voices (<< >> or \\), and nothing before or after the \relative block.
```
---

## 🔄 Run 1/20

### 🎵 Model Output
```lilypond

```
⏱️ Time: 28.30s | Tokens: 4426

---

## 🔄 Run 2/20

...

---

## 🔄 Run 20/20

### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
\key c \major
\time 4/4
\tempo 4 = 100
c4 d4 e4 f4 |
\grace { g8 } c4 d4 e4 f4 |
\acciaccatura { a8 } c4 d4 e4 f4 |
\appoggiatura { b8 } c4 d4 e4 f4 |
\tuplet 3/2 { c8 d e } f4 g4 a4 |
\times 2/3 { c8 d e } f4 g4 a4 |
\ottava 8 { c4 d4 e4 f4 } |
\transpose c c' { c4 d4 e4 f4 } |
}
```
⏱️ Time: 20.58s | Tokens: 4233

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\prompt_lab\prompt_lab_outputs\raw\expand_music_functions_off`